Load and explore the dataset

In [2]:
import bm25s
import pandas as pd 
import json 
import re

In [4]:
#### load data
meta_articles = pd.read_parquet("parquet_data/corpus_nl.parquet")
meta_qa = pd.read_parquet("parquet_data/qa_nl.parquet")
qrels = pd.read_parquet("parquet_data/qrels_nl.parquet")

In [5]:
num_words = meta_qa.answer.str.split().str.len().tolist()

In [6]:
pd.Series(num_words).describe()

count    1435.000000
mean      245.196516
std       117.630377
min        12.000000
25%       159.000000
50%       225.000000
75%       314.500000
max       720.000000
dtype: float64

In [8]:
### the questions contain 23 acronyms
acronym_pattern = re.compile(
    r"\b(?:"
        r"(?:[A-ZÁÀÉÈËÏÖÜÓÚ]{1,3}\.){2,}"   # A.O.W. B.T.W. V.N.
        r"|"
        r"[A-ZÁÀÉÈËÏÖÜÓÚ]{2,}"              # NAVO RIVM AOW COVID-19
        r")\b"
)
acronyms_query = (
    meta_qa["question"]
    .apply(lambda text: acronym_pattern.findall(text))
)

acronyms_query = sorted(set(acr for lst in acronyms_query for acr in lst))

print(acronyms_query)

['APA', 'CSR', 'DAVO', 'EPC', 'EU', 'GIB', 'GPMI', 'IT', 'IVT', 'MI', 'OCMW', 'RIZIV', 'RMI', 'RVA', 'THAB']


## BM25
For each token in the query, BM25 computes a score contribution based on:\
How often that token appears in this particular document (term frequency)\
How rare that token is across all documents (IDF)\
The document's length, normalized against the average document length in the collection (so a short document matching a token isn't unfairly penalized compared to a long one, and vice versa)\

In [ ]:
### French tokenization pattern
#TOKEN_PATTERN = r"""(?xu)
        # Acronyms with dots: C.E.D.H., U.E.
        #(?:[A-ZÀÂÄÇÉÈÊËÎÏÔÖÙÛÜŸŒ]{1,3}\.){2,}|

        # Acronyms: CNIL, RGPD, ONU, UE, GPT-4
       #[A-ZÀÂÄÇÉÈÊËÎÏÔÖÙÛÜŸŒ]{2,}[0-9-]*|
        
        # Legal references: L123-4, R4321-1
        #[LR]\d+(?:-\d+)+|

        # Apostrophe words:
        # l'article, d'une, qu'il
        #[A-Za-zÀ-ÖØ-öø-ÿŒœ]+['’-][A-Za-zÀ-ÖØ-öø-ÿŒœ]+|

        # Hyphenated words:
        # non-respect, pré-contentieux
        #[A-Za-zÀ-ÖØ-öø-ÿŒœ]+(?:-[A-Za-zÀ-ÖØ-öø-ÿŒœ0-9]+)+|

        # Normal French words
        #[A-Za-zÀ-ÖØ-öø-ÿŒœ]+|

        # Numbers and dates
        #\d+(?:[./-]\d+)*
    #"""

In [ ]:
TOKEN_PATTERN = r"""(?xu)
        (?:[A-ZÁÀÉÈËÏÖÜÓÚ]{1,3}\.){2,}|

        [A-ZÁÀÉÈËÏÖÜÓÚ]{2,}[0-9-]*|

        [A-Za-zÀ-ÖØ-öø-ÿ]*['’][A-Za-zÀ-ÖØ-öø-ÿ]+|

        [A-Za-zÀ-ÖØ-öø-ÿ]+(?:-[A-Za-zÀ-ÖØ-öø-ÿ0-9]+)+|

        [A-Za-zÀ-ÖØ-öø-ÿ]+|

        \d+(?:[./:-]\d+)*
    """

In [11]:
art_ids = meta_articles["id"].tolist()
art_texts = meta_articles["article"].tolist()

In [13]:
###BM25
tokens = bm25s.tokenize(art_texts, stopwords="dutch", token_pattern= TOKEN_PATTERN)
retriever = bm25s.BM25() # method='lucene' by default
retriever.index(tokens)

In [20]:
art_ids = meta_articles["id"].astype(str).tolist()

In [23]:
#save the indexes in the "indexes/BM25"
from pathlib import Path
INDEX_DIR = Path("indexes/BM25")
INDEX_DIR.mkdir(parents=True, exist_ok=True)
retriever.save(str(INDEX_DIR))
(INDEX_DIR / "doc_ids.txt").write_text("\n".join(art_ids))

145821

In [16]:
for acronym in acronyms_query:
    if acronym.lower() in tokens.vocab.keys():
        print(True, acronym)

True EU
True IT
True OCMW
True RIZIV
True RVA


In [24]:
def search_bm25(query: str, k: int = 10) -> list[tuple[str, float]]:
    """Return the top-k (art_id, score) pairs for a query."""
    query_tokens = bm25s.tokenize([query], stopwords="fr")
    #print(query_tokens)
    indices, scores = retriever.retrieve(query_tokens, k=k)
    
    # indices[0] is a numpy array of integer positions in doc_ids.
    return [
        (art_ids[i], float(scores[0][j])) for j, i in enumerate(indices[0].tolist())
    ]

In [ ]:
query = meta_qa.question[0]
print(f"\nQuery: {query}\n")
for i, (art_id, score) in enumerate(search_bm25(query1, k=5), 1):
    text = meta_articles.loc[meta_articles["id"] == art_id, "article"].values[0]
    print(f"{i}. [{score:6.2f}] {art_id}  {text}")

In [ ]:
### test code from retriever.py
from retrievers import BM25Retriever
bm_retriever = BM25Retriever()
search_res = bm_retriever.search(query1, k=5)
print(search_res)


In [ ]:
from retrievers import DocsRetriever
doc_retriever = DocsRetriever(search_res)
print(len(doc_retriever.documents()))

In [ ]:
print(doc_retriever.corpus.loc[doc_retriever.corpus["id"] == 26710, "article"].item())

Test on query decomposition

In [ ]:
query1 = "Ik wil scheiden wegens onherstelbare ontwrichting. Kan ik een tijdslimiet vastleggen voor de alimentatie?"
query2 = "Mijn wettelijk samenwonende partner is overleden. Wat zijn mijn rechten in zijn nalatenschap?"
query3 = "Ik ben ouder. Kan ik de alimentatie indexeren? Hoe bereken ik deze indexering?"

In [ ]:
from retrievers import BM25Retriever
bm_retriever = BM25Retriever()
search_res = bm_retriever.search(query1, k=5)
print(search_res)

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
                                                     

[('3600', 11.280284881591797), ('27155', 9.44019889831543), ('3602', 8.26412582397461), ('3601', 5.039328098297119), ('26022', 4.929961204528809)]
